# K-Nearest Neighbors (KNN) Classification

Instance-based, non-parametric algorithm that classifies data points based on majority vote of their k nearest neighbors.

## Topics Covered
1. KNN Algorithm Intuition
2. Effect of K value on decision boundary
3. Distance Metrics (Euclidean, Manhattan, Minkowski)
4. Feature Scaling importance
5. Finding Optimal K using Elbow Method
6. KNN for Regression

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Explore Data

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Pairplot
sns.pairplot(df, hue='species', diag_kind='kde', palette='Set2')
plt.suptitle('Iris Dataset - Feature Distribution', y=1.02)
plt.show()

## 2. Data Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling — CRITICAL for KNN (distance-based algorithm)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

## 3. KNN Classification

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)  # p=2 → Euclidean
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
disp.plot(cmap='Blues')
plt.title('KNN Confusion Matrix (k=5)')
plt.show()

## 4. Finding Optimal K — Elbow Method

In [ ]:
k_range = range(1, 31)
train_scores = []
test_scores = []
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))
    cv = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(cv.mean())

plt.figure(figsize=(12, 5))
plt.plot(k_range, train_scores, 'o-', label='Train Accuracy', color='steelblue')
plt.plot(k_range, test_scores, 's-', label='Test Accuracy', color='coral')
plt.plot(k_range, cv_scores, '^-', label='CV Accuracy (5-fold)', color='green')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Accuracy')
plt.title('Finding Optimal K — Elbow Method')
plt.xticks(k_range)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(cv_scores)]
print(f"\nOptimal K (by CV): {best_k} with accuracy {max(cv_scores):.4f}")

## 5. Effect of K on Decision Boundary

In [ ]:
# Use only 2 features for visualization
X_2d = X_train_scaled[:, :2]
X_test_2d = X_test_scaled[:, :2]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])

for ax, k in zip(axes, [1, 5, 15, 29]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_2d, y_train)
    
    # Create mesh grid
    h = 0.05
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_train, cmap=cmap_bold, edgecolor='k', s=20)
    ax.set_title(f'K = {k} (Acc={knn.score(X_test_2d, y_test):.2f})')

plt.suptitle('Effect of K on Decision Boundary', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Distance Metrics Comparison

In [ ]:
metrics = {
    'Euclidean (p=2)': {'metric': 'minkowski', 'p': 2},
    'Manhattan (p=1)': {'metric': 'minkowski', 'p': 1},
    'Chebyshev': {'metric': 'chebyshev'},
}

print(f"{'Metric':<20} {'Accuracy':>10} {'CV Mean':>10}")
print("-" * 42)

for name, params in metrics.items():
    knn = KNeighborsClassifier(n_neighbors=best_k, **params)
    knn.fit(X_train_scaled, y_train)
    acc = knn.score(X_test_scaled, y_test)
    cv = cross_val_score(knn, X_train_scaled, y_train, cv=5).mean()
    print(f"{name:<20} {acc:>10.4f} {cv:>10.4f}")

## 7. Importance of Feature Scaling

In [ ]:
# Without scaling
knn_no_scale = KNeighborsClassifier(n_neighbors=5)
knn_no_scale.fit(X_train, y_train)
acc_no_scale = knn_no_scale.score(X_test, y_test)

# With scaling
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
acc_scaled = knn_scaled.score(X_test_scaled, y_test)

print(f"Without Scaling: {acc_no_scale:.4f}")
print(f"With Scaling:    {acc_scaled:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Without Scaling', 'With Scaling'], [acc_no_scale, acc_scaled],
              color=['coral', 'steelblue'])
ax.set_ylabel('Accuracy')
ax.set_title('Impact of Feature Scaling on KNN')
ax.set_ylim(0.8, 1.05)
for bar, val in zip(bars, [acc_no_scale, acc_scaled]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

## Key Takeaways

| Aspect | Details |
|--------|---------|  
| **Algorithm Type** | Instance-based, lazy learner (no training phase) |
| **Feature Scaling** | REQUIRED — distances are scale-sensitive |
| **Optimal K** | Use cross-validation + elbow method |
| **Small K** | Complex boundary, overfitting |
| **Large K** | Smooth boundary, underfitting |
| **Computational Cost** | O(n) per prediction (scales with dataset size) |
| **Curse of Dimensionality** | Performance degrades in high dimensions |